In [ ]:
%cd /home/sazhang/Tokenized-Neural-Rendering

/home/sazhang/Tokenized-Neural-Rendering


In [9]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from model.renderer import TransformerRF
from model.tri_token_model import TriangleEncoder
from pipeline.simple_dataset import SingleObjectDataset


In [7]:
# --- Quick Setup ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Instantiate Decoder
decoder = TransformerRF(token_dim=128, num_layers=1, nhead=2).to(device)

# 2. Create a "Dummy Encoder" (just a learnable vector)
# This mimics what the PointNet WOULD output for a single object
# We use nn.Parameter so the optimizer can update it
dummy_object_code = nn.Parameter(torch.randn(1, 128).to(device))

# 3. Optimizer (Optimize BOTH the decoder weights AND the object code)
optimizer = torch.optim.Adam([
    {'params': decoder.parameters()},
    {'params': dummy_object_code} 
], lr=1e-3)

/home/sazhang/miniconda3/envs/tokenized_nr/lib/python3.11/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


In [ ]:

# 1. Initialize Dataset
dataset = SingleObjectDataset(npz_path='experiment_data.npz', device='cuda')

# 2. Initialize DataLoader
dataloader = DataLoader(
    dataset, 
    batch_size=32,  # Number of rays per training step
    shuffle=True,     # Mixes pixels from different views/regions
    num_workers=0     # Keep 0 if data is already on GPU (device='cuda')
)

Loaded 16384 rays to GPU memory.


In [17]:
dataset[0]

{'rays_o': tensor([1.6891, 0.8594, 0.6390], device='cuda:0'),
 'rays_d': tensor([-0.9848,  0.1099,  0.1348], device='cuda:0'),
 'rgb': tensor([1., 1., 1.], device='cuda:0')}

In [ ]:
# --- Training Loop Pseudo-code ---
criterion = nn.MSELoss()
for batch in dataloader:
    print(batch)
    w_out = - batch['rays_d'].to(device)   # [Batch, 3]  Note: Camera looks down -Z, so w_out is opposite
    target_rgb = batch['rgb'].to(device)   # [Batch, 3]

    # Expand code to match batch size
    batch_code = dummy_object_code.expand(w_out.shape[0], -1)

    # Forward
    pred_rgb = decoder(batch_code, w_out)

    # Loss
    loss = criterion(pred_rgb, target_rgb)
    loss.backward()
    optimizer.step()

TypeError: 'DataLoader' object is not subscriptable